# Final outer-test evaluation: primary full-M0 models and paired weather ablation

Notebook 06 full-M0 calibrated models are the primary RQ2/RQ4 models and are evaluated on all eligible outer-test rows. Notebook 08 paired M0/M1 bundles answer RQ3 on identical complete-weather rows. Natural unseen-household results are supplementary and exploratory. No model fitting, calibration fitting, feature selection, or threshold selection occurs here.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, brier_score_loss,
                             confusion_matrix, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score, accuracy_score)

ROOT=Path.cwd().resolve()
while not (ROOT/"data"/"processed").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/"data"/"processed").exists(): raise RuntimeError("Could not locate project root")
DATA_DIR=ROOT/"data"/"processed"
TABLE_DIR=ROOT/"reports"/"tables"/"08b_final_test_evaluation"
FIGURE_DIR=ROOT/"reports"/"figures"/"08b_final_test_evaluation"
MODEL_DIR=ROOT/"models"/"08_weather_model_comparison"
for d in (TABLE_DIR,FIGURE_DIR): d.mkdir(parents=True,exist_ok=True)
MASTER_PATH=DATA_DIR/"04_master_split.csv"
PAIR_AUDIT_PATH=DATA_DIR/"07_m0_m1_pairing_audit.csv"
LOCK_MANIFEST_PATH=ROOT/"reports"/"tables"/"08_weather_model_comparison"/"08_paired_weather_experiment_manifest.json"
DATA_PATHS={"P95_M0":DATA_DIR/"05_model_ready_m0_p95.csv","P90_M0":DATA_DIR/"05_model_ready_m0_p90.csv",
            "P95_M1":DATA_DIR/"07_model_ready_m1_p95.csv","P90_M1":DATA_DIR/"07_model_ready_m1_p90.csv"}
N_BOOTSTRAP=1000
BOOTSTRAP_SEED=20261007

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
 return h.hexdigest()
def probability_logit(probability):
 p=np.clip(np.asarray(probability,dtype=float),1e-6,1-1e-6)
 return np.log(p/(1-p)).reshape(-1,1)
def numeric_x(frame,features):
 return frame[features].apply(lambda c:pd.to_numeric(c,errors="coerce").astype("float64"))


## 1. Verify the frozen experiment and select outer-test rows

The lock manifest must confirm that the models, calibrators, and thresholds were produced without outer-test use. The pairing audit selects the complete-weather rows, while the master split supplies cohort labels. Model-ready parsing is restricted to those reserved test IDs; internal training, calibration, and threshold-selection labels are not read.

In [ ]:
for path in [MASTER_PATH,PAIR_AUDIT_PATH,LOCK_MANIFEST_PATH,*DATA_PATHS.values()]:
 if not path.exists(): raise FileNotFoundError(f"Required final-evaluation input missing: {path}")
lock=read_json(LOCK_MANIFEST_PATH)
if not lock.get("validation_passed",False) or lock.get("outer_test_labels_loaded") or lock.get("outer_test_scored"):
 raise ValueError("Notebook 08 handoff is not a valid internal-only model lock")
master=pd.read_csv(MASTER_PATH,dtype={"row_id":"string","household_id":"string"})
master["target_date"]=pd.to_datetime(master["target_date"],errors="raise")
pair=pd.read_csv(PAIR_AUDIT_PATH,dtype={"row_id":"string","household_id":"string"})
if master.row_id.duplicated().any() or pair.row_id.duplicated().any(): raise ValueError("Row IDs must be unique in split and weather audit")
complete_ids=set(pair.loc[pair.m1_weather_complete.fillna(False).astype(bool),"row_id"].astype(str))
test_master=master.loc[master.outer_split.eq("test") & master.row_id.astype(str).isin(complete_ids),["row_id","household_id","target_date","outer_split","test_cohort"]].copy()
if test_master.empty: raise ValueError("No complete-weather outer-test rows are available")
if set(test_master.test_cohort.unique())!={"test_seen","test_unseen"}: raise ValueError("Expected both test_seen and natural test_unseen cohorts")
test_ids=set(test_master.row_id.astype(str))

def read_test_rows(path,expected_ids,features):
 rowids=pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"}).row_id
 positions=np.flatnonzero(rowids.isin(expected_ids).to_numpy())
 found=set(rowids.iloc[positions].astype(str))
 if found!=expected_ids: raise ValueError(f"{path.name}: test ID coverage differs from master/pair audit")
 keep=set(positions.tolist()); columns=list(dict.fromkeys(["row_id","household_id","measurement_date","target",*features]))
 frame=pd.read_csv(path,usecols=columns,dtype={"row_id":"string","household_id":"string"},skiprows=lambda line:line>0 and (line-1) not in keep)
 if frame.row_id.duplicated().any() or set(frame.row_id.astype(str))!=expected_ids: raise ValueError(f"{path.name}: selective test parser returned unexpected rows")
 frame["measurement_date"]=pd.to_datetime(frame["measurement_date"],errors="raise")
 frame=frame.merge(test_master,on="row_id",how="left",validate="one_to_one",suffixes=("","_master"))
 if frame.household_id.astype(str).ne(frame.household_id_master.astype(str)).any(): raise ValueError(f"{path.name}: household ID differs from frozen master")
 if not frame.measurement_date.eq(frame.target_date).all(): raise ValueError(f"{path.name}: target date differs from frozen master")
 if not frame.outer_split.eq("test").all(): raise ValueError(f"{path.name}: non-test rows parsed")
 return frame.drop(columns=["household_id_master"])

bundles={}; test_frames={}; predictions={}; metric_rows=[]; cohort_rows=[]
for key,path in DATA_PATHS.items():
 bundle_path=MODEL_DIR/f"{key.lower()}_paired_weather.joblib"
 if not bundle_path.exists(): raise FileNotFoundError(f"Missing locked model bundle: {bundle_path}")
 bundle=joblib.load(bundle_path)
 target,dataset=key.split("_")
 if bundle["target"]!=target or bundle["dataset"]!=dataset or bundle.get("model_role")!="controlled_weather_ablation" or bundle.get("outer_test_used") is not False: raise ValueError(f"Invalid bundle metadata: {bundle_path.name}")
 if bundle["threshold"]!=lock["models"][key]["threshold"]: raise ValueError(f"Threshold differs from lock manifest for {key}")
 frame=read_test_rows(path,test_ids,bundle["features"])
 bundles[key]=bundle; test_frames[key]=frame

# M0/M1 and P95/P90 must share row, household, date, cohort, and target assignments.
reference=test_frames["P95_M0"]
for key,frame in test_frames.items():
 for col in ("row_id","household_id","measurement_date","test_cohort"):
  if not reference[col].reset_index(drop=True).equals(frame[col].reset_index(drop=True)): raise ValueError(f"Final test pairing differs in {key}.{col}")

for target in ("P95","P90"):
 if not test_frames[f"{target}_M0"].target.reset_index(drop=True).equals(test_frames[f"{target}_M1"].target.reset_index(drop=True)):
  raise ValueError(f"{target}: M0/M1 test labels differ")

for key,frame in test_frames.items():
 bundle=bundles[key]
 raw=bundle["base_estimator"].predict_proba(numeric_x(frame,bundle["features"]))[:,1]
 calibrated=bundle["platt_calibrator"].predict_proba(probability_logit(raw))[:,1]
 if not np.isfinite(calibrated).all() or ((calibrated<0)|(calibrated>1)).any(): raise ValueError(f"Invalid probabilities from locked bundle {key}")
 predictions[key]={"raw":raw,"calibrated":calibrated}

# Persist frozen row-level scores so Notebook 09 can select local SHAP cases without rescoring.
row_prediction_frames=[]
for key,frame in test_frames.items():
 target,dataset=key.split("_"); bundle=bundles[key]; probability=predictions[key]["calibrated"]
 scored=frame[["row_id","household_id","measurement_date","target","test_cohort"]].copy()
 scored["target_name"]=target; scored["dataset"]=dataset; scored["raw_base_probability"]=predictions[key]["raw"]; scored["calibrated_probability"]=probability
 scored["locked_threshold"]=bundle["threshold"]
 scored["predicted_label"]=(probability>=bundle["threshold"]).astype("int8")
 row_prediction_frames.append(scored)
row_predictions=pd.concat(row_prediction_frames,ignore_index=True)
row_prediction_path=TABLE_DIR/"08b_final_test_row_predictions.csv"
row_predictions.to_csv(row_prediction_path,index=False)
if len(row_predictions)!=len(test_ids)*len(test_frames) or row_predictions.duplicated(["target_name","dataset","row_id"]).any():
 raise ValueError("Row-level final-test prediction export failed its one-row-per-model/ID check")

cohort_summary=(reference.groupby("test_cohort",as_index=False).agg(rows=("row_id","size"),households=("household_id","nunique"),start_date=("measurement_date","min"),end_date=("measurement_date","max")))
cohort_summary.to_csv(TABLE_DIR/"08b_test_cohort_summary.csv",index=False)
display(cohort_summary)
print("Outer-test IDs parsed:",f"{len(test_ids):,}","| internal-stage target rows parsed: 0")


In [ ]:
# Primary full-M0 models (Notebook 06) use every eligible M0 outer-test row.
PRIMARY_MODEL_DIR=ROOT/"models"/"06_calibrated_m0"
PRIMARY_LOCK_PATH=ROOT/"reports"/"tables"/"06_calibration_evaluation"/"06_calibration_and_threshold_manifest.json"
primary_lock=read_json(PRIMARY_LOCK_PATH)
if not primary_lock.get("validation_passed") or primary_lock.get("outer_test_used_for_fit_calibration_or_threshold"):
 raise ValueError("Notebook 06 primary-model lock is invalid")
full_test_master=master.loc[master.outer_split.eq("test"),["row_id","household_id","target_date","outer_split","test_cohort"]].copy()
primary_test_ids=set(full_test_master.row_id.astype(str))
if not primary_test_ids or full_test_master.row_id.duplicated().any(): raise ValueError("Frozen full-M0 test IDs are invalid")
def read_primary_test_rows(path,expected_ids,features):
 rowids=pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"}).row_id
 positions=np.flatnonzero(rowids.isin(expected_ids).to_numpy())
 if set(rowids.iloc[positions].astype(str))!=expected_ids: raise ValueError(f"{path.name}: primary IDs differ from frozen split")
 keep=set(positions.tolist()); cols=list(dict.fromkeys(["row_id","household_id","measurement_date","target",*features]))
 frame=pd.read_csv(path,usecols=cols,dtype={"row_id":"string","household_id":"string"},skiprows=lambda line:line>0 and (line-1) not in keep)
 if frame.row_id.duplicated().any() or set(frame.row_id.astype(str))!=expected_ids: raise ValueError("Unexpected primary test rows")
 frame["measurement_date"]=pd.to_datetime(frame.measurement_date,errors="raise")
 frame=frame.merge(full_test_master,on="row_id",how="left",validate="one_to_one",suffixes=("","_master"))
 if frame.household_id.astype(str).ne(frame.household_id_master.astype(str)).any() or not frame.measurement_date.eq(frame.target_date).all() or not frame.outer_split.eq("test").all(): raise ValueError("Primary rows differ from frozen split")
 return frame.drop(columns=["household_id_master"])
primary_bundles={}; primary_test_frames={}; primary_predictions={}
for target in ("P95","P90"):
 bundle=joblib.load(PRIMARY_MODEL_DIR/f"{target.lower()}_calibrated_m0.joblib")
 if bundle.get("model_role")!="primary_full_m0" or not {"RQ2","RQ4"}.issubset(set(bundle.get("research_questions",[]))) or bundle.get("outer_test_used_for_fit_calibration_or_threshold") is not False: raise ValueError(f"Invalid primary bundle: {target}")
 manifest_model=primary_lock["selected_models"][target]
 if bundle["threshold"]!=manifest_model["threshold"] or manifest_model["model_role"]!="primary_full_m0": raise ValueError("Primary bundle differs from Notebook 06 lock")
 frame=read_primary_test_rows(DATA_PATHS[f"{target}_M0"],primary_test_ids,bundle["features"])
 raw=bundle["base_estimator"].predict_proba(numeric_x(frame,bundle["features"]))[:,1]
 prob=bundle["platt_calibrator"].predict_proba(probability_logit(raw))[:,1]
 if not np.isfinite(prob).all() or ((prob<0)|(prob>1)).any(): raise ValueError("Invalid primary probabilities")
 primary_bundles[target]=bundle; primary_test_frames[target]=frame; primary_predictions[target]={"raw":raw,"calibrated":prob}
primary_prediction_frames=[]
for target,frame in primary_test_frames.items():
 bundle=primary_bundles[target]; prob=primary_predictions[target]["calibrated"]
 scored=frame[["row_id","household_id","measurement_date","target","test_cohort"]].copy()
 scored["target_name"]=target; scored["dataset"]="PRIMARY_FULL_M0"; scored["raw_base_probability"]=primary_predictions[target]["raw"]; scored["calibrated_probability"]=prob
 scored["locked_threshold"]=bundle["threshold"]; scored["predicted_label"]=(prob>=bundle["threshold"]).astype("int8")
 primary_prediction_frames.append(scored)
row_predictions=pd.concat([row_predictions,*primary_prediction_frames],ignore_index=True)
row_predictions.to_csv(row_prediction_path,index=False)
if row_predictions.duplicated(["target_name","dataset","row_id"]).any(): raise ValueError("Duplicate prediction records")
print("Scored primary full-M0 test rows:",f"{len(primary_test_ids):,}","| paired weather rows:",f"{len(test_ids):,}")
def metric_values(y,probability,threshold,weights=None):
 y=np.asarray(y,dtype="int8"); pred=(np.asarray(probability)>=threshold).astype("int8")
 tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1],sample_weight=weights).ravel()
 return {"pr_auc":average_precision_score(y,probability,sample_weight=weights),"roc_auc":roc_auc_score(y,probability,sample_weight=weights),
  "brier_score":brier_score_loss(y,probability,sample_weight=weights),"log_loss":log_loss(y,probability,labels=[0,1],sample_weight=weights),
  "precision":precision_score(y,pred,zero_division=0,sample_weight=weights),"recall":recall_score(y,pred,zero_division=0,sample_weight=weights),
  "f1":f1_score(y,pred,zero_division=0,sample_weight=weights),"balanced_accuracy":balanced_accuracy_score(y,pred,sample_weight=weights),
  "accuracy":accuracy_score(y,pred,sample_weight=weights),"tn":tn,"fp":fp,"fn":fn,"tp":tp}


## 2. Primary full-M0 evaluation for RQ2 and RQ4

The primary models are evaluated on all eligible outer-test rows, including dates without complete archived weather. The constant-probability baseline uses only model-development prevalence. Reliability and calibration slope/intercept are evaluation diagnostics; no recalibration or threshold changes follow. P95 risk deciles and gains use the primary full-M0 calibrated probabilities.

In [ ]:
import scipy.optimize

primary_metric_rows=[]; primary_calibration_rows=[]; primary_reliability_frames={}; primary_development_prevalence={}
for target,frame in primary_test_frames.items():
 bundle=primary_bundles[target]; prob=primary_predictions[target]["calibrated"]
 dev_ids=set(master.loc[master.internal_split.eq("model_development"),"row_id"].astype(str))
 source=DATA_DIR/f"05_model_ready_m0_{target.lower()}.csv"
 train_labels=pd.read_csv(source,usecols=["row_id","target"],dtype={"row_id":"string","target":"int8"})
 train_labels=train_labels.loc[train_labels.row_id.astype(str).isin(dev_ids),"target"].astype("int8")
 if len(train_labels)!=len(dev_ids): raise ValueError(f"{target}: development rows differ from frozen split")
 p_dev=float(train_labels.mean()); primary_development_prevalence[target]=p_dev
 for cohort in ("test_seen","test_unseen"):
  mask=frame.test_cohort.eq(cohort).to_numpy()
  if not mask.any(): continue
  y=frame.loc[mask,"target"].astype("int8").to_numpy(); p=prob[mask]
  values=metric_values(y,p,bundle["threshold"])
  base_brier=brier_score_loss(y,np.full(len(y),p_dev)); base_logloss=log_loss(y,np.full(len(y),p_dev),labels=[0,1])
  primary_metric_rows.append({"target":target,"model_role":"primary_full_m0","research_questions":"RQ2,RQ4","cohort":cohort,
   "rows":int(mask.sum()),"households":int(frame.loc[mask,"household_id"].nunique()),"events":int(y.sum()),
   "test_prevalence_descriptive":float(y.mean()),"development_prevalence_baseline":p_dev,
   "development_prevalence_baseline_brier":base_brier,"development_prevalence_baseline_log_loss":base_logloss,
   "brier_skill_score":1-values["brier_score"]/base_brier if base_brier>0 else np.nan,"threshold":bundle["threshold"],**values})
 mask=frame.test_cohort.eq("test_seen").to_numpy()
 y=frame.loc[mask,"target"].astype("int8").to_numpy(); p=prob[mask]
 logits=np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
 def objective(params):
  z=params[0]+params[1]*logits
  return float(np.logaddexp(0,z).sum()-np.dot(y,z))
 fit=scipy.optimize.minimize(objective,np.array([0.0,1.0]),method="BFGS")
 if not np.isfinite(fit.fun): raise RuntimeError("Calibration diagnostic failed")
 primary_calibration_rows.append({"target":target,"cohort":"test_seen","calibration_intercept":float(fit.x[0]),
  "calibration_slope":float(fit.x[1]),"ideal_intercept":0.0,"ideal_slope":1.0,"rows":len(y),
  "method":"unpenalized final-test logistic recalibration diagnostic; no refit applied"})
 bin_id=pd.qcut(pd.Series(p).rank(method="first"),q=10,labels=False,duplicates="drop")
 rel=pd.DataFrame({"bin":bin_id.to_numpy(),"probability":p,"observed":y})
 rel=rel.groupby("bin",as_index=False).agg(rows=("observed","size"),mean_predicted_probability=("probability","mean"),observed_event_rate=("observed","mean"),events=("observed","sum"))
 rel.insert(0,"target",target); rel["model_role"]="primary_full_m0"; primary_reliability_frames[target]=rel
primary_metrics=pd.DataFrame(primary_metric_rows)
primary_metrics.to_csv(TABLE_DIR/"08b_primary_full_m0_metrics.csv",index=False)
primary_calibration=pd.DataFrame(primary_calibration_rows)
primary_calibration.to_csv(TABLE_DIR/"08b_primary_full_m0_calibration_diagnostics.csv",index=False)
primary_reliability=pd.concat(primary_reliability_frames.values(),ignore_index=True)
primary_reliability.to_csv(TABLE_DIR/"08b_primary_full_m0_reliability_bins.csv",index=False)
display(primary_metrics); display(primary_calibration)
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 rel=primary_reliability.loc[primary_reliability.target.eq(target)]
 ax.plot([0,1],[0,1],"--",color="gray",label="Ideal")
 ax.plot(rel.mean_predicted_probability,rel.observed_event_rate,"o-",label="Primary full M0")
 ax.set(title=f"{target}: primary full-M0 reliability on test_seen",xlabel="Mean calibrated probability",ylabel="Observed event rate",xlim=(0,1),ylim=(0,1)); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_primary_full_m0_reliability.png",dpi=160,bbox_inches="tight"); plt.close(fig)

frame=primary_test_frames["P95"]; mask=frame.test_cohort.eq("test_seen").to_numpy()
risk=pd.DataFrame({"row_id":frame.loc[mask,"row_id"].astype(str).to_numpy(),"household_id":frame.loc[mask,"household_id"].astype(str).to_numpy(),
 "target":frame.loc[mask,"target"].astype("int8").to_numpy(),"probability":primary_predictions["P95"]["calibrated"][mask]})
risk=risk.sort_values(["probability","row_id"],ascending=[False,True]).reset_index(drop=True)
risk["risk_decile"]=pd.qcut(risk.index.to_series().rank(method="first",ascending=False),q=10,labels=False,duplicates="drop")+1
risk_decile=risk.groupby("risk_decile",as_index=False).agg(rows=("target","size"),events=("target","sum"),event_rate=("target","mean"),mean_predicted_probability=("probability","mean"))
risk_decile["share_of_all_events"]=risk_decile.events/risk.target.sum(); risk_decile["overall_event_rate"]=float(risk.target.mean()); risk_decile["lift"]=risk_decile.event_rate/risk_decile.overall_event_rate
risk_decile["model_role"]="primary_full_m0"; risk_decile["target"]="P95"
risk_decile.to_csv(TABLE_DIR/"08b_primary_p95_risk_deciles.csv",index=False)
gain_rows=[]
for pct in range(10,101,10):
 n=int(np.ceil(len(risk)*pct/100)); events=int(risk.head(n).target.sum())
 gain_rows.append({"top_percent_prioritized":pct,"rows_prioritized":n,"events_captured":events,"fraction_of_all_events_captured":events/risk.target.sum(),"random_baseline_fraction":pct/100})
gains=pd.DataFrame(gain_rows); gains.to_csv(TABLE_DIR/"08b_primary_p95_cumulative_gains.csv",index=False)
fig,ax=plt.subplots(figsize=(7,6))
ax.plot(gains.top_percent_prioritized,gains.fraction_of_all_events_captured,"o-",label="Primary P95 full M0")
ax.plot([0,100],[0,1],"--",color="gray",label="Random prioritization")
ax.set(title="P95 cumulative gains — primary full M0",xlabel="Household-days prioritized (%)",ylabel="P95 events captured (%)",xlim=(0,100),ylim=(0,1)); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_primary_p95_cumulative_gains.png",dpi=160,bbox_inches="tight"); plt.close(fig)
display(risk_decile); display(gains)

## 3. Controlled paired weather ablation for RQ3

Notebook 08 paired M0 and M1 bundles are evaluated on identical complete-weather outer-test rows. Weather effects are reported only as paired M1−M0 differences. The paired M1 model is never compared with primary full-M0.

In [ ]:
def metric_values(y,probability,threshold,weights=None):
 y=np.asarray(y,dtype="int8"); pred=(np.asarray(probability)>=threshold).astype("int8")
 tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1],sample_weight=weights).ravel()
 return {"pr_auc":average_precision_score(y,probability,sample_weight=weights),
         "roc_auc":roc_auc_score(y,probability,sample_weight=weights),
         "brier_score":brier_score_loss(y,probability,sample_weight=weights),
         "log_loss":log_loss(y,probability,labels=[0,1],sample_weight=weights),
         "precision":precision_score(y,pred,zero_division=0,sample_weight=weights),
         "recall":recall_score(y,pred,zero_division=0,sample_weight=weights),
         "f1":f1_score(y,pred,zero_division=0,sample_weight=weights),
         "balanced_accuracy":balanced_accuracy_score(y,pred,sample_weight=weights),
         "accuracy":accuracy_score(y,pred,sample_weight=weights),
         "tn":tn,"fp":fp,"fn":fn,"tp":tp}

for key,frame in test_frames.items():
 target,dataset=key.split("_"); bundle=bundles[key]; probability=predictions[key]["calibrated"]
 for cohort in ("test_seen","test_unseen"):
  mask=frame.test_cohort.eq(cohort).to_numpy()
  if not mask.any(): continue
  y=frame.loc[mask,"target"].astype("int8").to_numpy()
  if np.unique(y).size<2: raise ValueError(f"{key} {cohort} does not contain both classes; ranking metrics are undefined")
  values=metric_values(y,probability[mask],bundle["threshold"])
  metric_rows.append({"target":target,"dataset":dataset,"model_role":"controlled_weather_ablation","research_questions":"RQ3","cohort":cohort,"cohort_interpretation":"primary known-household future period" if cohort=="test_seen" else "exploratory natural unseen-household cohort",
                      "rows":int(mask.sum()),"households":int(frame.loc[mask,"household_id"].nunique()),"events":int(y.sum()),"prevalence":float(y.mean()),
                      "no_skill_pr_auc_reference":float(y.mean()),"threshold":bundle["threshold"],**values})
metrics=pd.DataFrame(metric_rows)
metrics.to_csv(TABLE_DIR/"08b_final_test_metrics.csv",index=False)
display(metrics)

# Paired M1-M0 point differences use exactly the same test rows and labels.
delta_rows=[]
for target in ("P95","P90"):
 for cohort in ("test_seen","test_unseen"):
  mask=test_frames[f"{target}_M0"].test_cohort.eq(cohort).to_numpy()
  if not mask.any(): continue
  y=test_frames[f"{target}_M0"].loc[mask,"target"].astype("int8").to_numpy()
  m0=metric_values(y,predictions[f"{target}_M0"]["calibrated"][mask],bundles[f"{target}_M0"]["threshold"])
  m1=metric_values(y,predictions[f"{target}_M1"]["calibrated"][mask],bundles[f"{target}_M1"]["threshold"])
  for metric in ("pr_auc","roc_auc","brier_score","log_loss","precision","recall","f1","balanced_accuracy"):
   delta_rows.append({"target":target,"model_role":"controlled_weather_ablation","research_questions":"RQ3","cohort":cohort,"cohort_interpretation":"primary paired test_seen" if cohort=="test_seen" else "exploratory natural unseen-household cohort",
                      "metric":metric,"m0":m0[metric],"m1":m1[metric],"delta_m1_minus_m0":m1[metric]-m0[metric],"rows":int(mask.sum()),
                      "interpretation":"positive favors M1" if metric in ("pr_auc","roc_auc","precision","recall","f1","balanced_accuracy") else "negative favors M1"})
differences=pd.DataFrame(delta_rows)
differences.to_csv(TABLE_DIR/"08b_paired_weather_differences.csv",index=False)
display(differences)


## 4. Household-cluster bootstrap confidence intervals

The paired intervals here resample households and use the same household multiplicities for paired M0 and M1. Separate primary full-M0 intervals are reported for the Notebook 06 models.

In [ ]:
rng=np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows=[]
M0_CI_METRICS=("pr_auc","roc_auc","brier_score","log_loss","precision","recall","f1","balanced_accuracy")
DELTA_CI_METRICS=("pr_auc","roc_auc","brier_score","log_loss")

for target in ("P95","P90"):
 ref=test_frames[f"{target}_M0"]; mask=ref.test_cohort.eq("test_seen").to_numpy()
 households=ref.loc[mask,"household_id"].astype(str).to_numpy(); household_codes,unique_households=pd.factorize(households,sort=True)
 if len(unique_households)<2: raise ValueError(f"Insufficient test_seen households for cluster bootstrap: {target}")
 y=ref.loc[mask,"target"].astype("int8").to_numpy()
 probs={ds:predictions[f"{target}_{ds}"]["calibrated"][mask] for ds in ("M0","M1")}
 thresholds={ds:bundles[f"{target}_{ds}"]["threshold"] for ds in ("M0","M1")}
 samples={f"{ds}:{metric}":[] for ds in ("M0",) for metric in M0_CI_METRICS}
 samples.update({f"DELTA:{metric}":[] for metric in DELTA_CI_METRICS})
 for _ in range(N_BOOTSTRAP):
  draws=rng.integers(0,len(unique_households),size=len(unique_households))
  cluster_multiplicity=np.bincount(draws,minlength=len(unique_households))
  weights=cluster_multiplicity[household_codes].astype(float)
  if weights.sum()==0: continue
  score_m0=metric_values(y,probs["M0"],thresholds["M0"],weights=weights)
  score_m1=metric_values(y,probs["M1"],thresholds["M1"],weights=weights)
  for metric in M0_CI_METRICS: samples[f"M0:{metric}"].append(score_m0[metric])
  for metric in DELTA_CI_METRICS: samples[f"DELTA:{metric}"].append(score_m1[metric]-score_m0[metric])
 for statistic,values in samples.items():
  lo,hi=np.quantile(values,[0.025,0.975])
  if statistic.startswith("M0:"):
   dataset="M0"; metric=statistic.split(":",1)[1]; point=metrics.loc[(metrics.target==target)&(metrics.dataset==dataset)&(metrics.cohort=="test_seen"),metric].iloc[0]
  else:
   dataset="M1-M0"; metric=statistic.split(":",1)[1]; point=differences.loc[(differences.target==target)&(differences.cohort=="test_seen")&(differences.metric==metric),"delta_m1_minus_m0"].iloc[0]
  bootstrap_rows.append({"target":target,"cohort":"test_seen","quantity":dataset,"metric":metric,"point_estimate":point,
                         "ci_95_lower":float(lo),"ci_95_upper":float(hi),"bootstrap_replicates":len(values),
                         "households":len(unique_households),"resampling_unit":"household; paired multiplicities for M1-M0" if dataset=="M1-M0" else "household"})
 print(f"{target}: completed {N_BOOTSTRAP:,} paired household-bootstrap replicates across {len(unique_households):,} test_seen households")

bootstrap_ci=pd.DataFrame(bootstrap_rows)
bootstrap_ci.to_csv(TABLE_DIR/"08b_household_bootstrap_confidence_intervals.csv",index=False)
display(bootstrap_ci)

# Separate household-bootstrap intervals for the primary Notebook 06 full-M0 models.
primary_bootstrap_rows=[]
primary_rng=np.random.default_rng(BOOTSTRAP_SEED+1)
for target,frame in primary_test_frames.items():
 mask=frame.test_cohort.eq("test_seen").to_numpy()
 households=frame.loc[mask,"household_id"].astype(str).to_numpy()
 codes,unique_households=pd.factorize(households,sort=True)
 y=frame.loc[mask,"target"].astype("int8").to_numpy()
 prob=primary_predictions[target]["calibrated"][mask]
 threshold=primary_bundles[target]["threshold"]
 samples={metric:[] for metric in M0_CI_METRICS}
 for _ in range(N_BOOTSTRAP):
  draws=primary_rng.integers(0,len(unique_households),size=len(unique_households))
  multiplicity=np.bincount(draws,minlength=len(unique_households))
  weights=multiplicity[codes].astype(float)
  scores=metric_values(y,prob,threshold,weights=weights)
  for metric in M0_CI_METRICS: samples[metric].append(scores[metric])
 for metric,values in samples.items():
  point=primary_metrics.loc[(primary_metrics.target==target)&(primary_metrics.cohort=="test_seen"),metric].iloc[0]
  lo,hi=np.quantile(values,[0.025,0.975])
  primary_bootstrap_rows.append({"target":target,"model_role":"primary_full_m0","research_questions":"RQ2,RQ4","cohort":"test_seen",
   "metric":metric,"point_estimate":point,"ci_95_lower":float(lo),"ci_95_upper":float(hi),
   "bootstrap_replicates":len(values),"households":len(unique_households),"resampling_unit":"household"})
primary_bootstrap_ci=pd.DataFrame(primary_bootstrap_rows)
primary_bootstrap_ci.to_csv(TABLE_DIR/"08b_primary_full_m0_household_bootstrap_ci.csv",index=False)
display(primary_bootstrap_ci)


## 5. Plot final confusion matrices and summarize interpretation

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig,axes=plt.subplots(2,2,figsize=(10,9))
for ax,(target,dataset) in zip(axes.flat,[("P95","M0"),("P95","M1"),("P90","M0"),("P90","M1")]):
 frame=test_frames[f"{target}_{dataset}"]; mask=frame.test_cohort.eq("test_seen").to_numpy()
 y=frame.loc[mask,"target"].astype("int8").to_numpy(); prob=predictions[f"{target}_{dataset}"]["calibrated"][mask]
 ConfusionMatrixDisplay.from_predictions(y,prob>=bundles[f"{target}_{dataset}"]["threshold"],labels=[0,1],values_format="d",ax=ax,colorbar=False)
 ax.set_title(f"{target} {dataset} — test_seen; threshold={bundles[f'{target}_{dataset}']['threshold']:.2f}")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_test_seen_confusion_matrices.png",dpi=160,bbox_inches="tight"); plt.show()

from sklearn.metrics import precision_recall_curve, roc_curve

# Ranking curves on the primary known-household test cohort.
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 for dataset in ("M0","M1"):
  frame=test_frames[f"{target}_{dataset}"]; mask=frame.test_cohort.eq("test_seen").to_numpy()
  y=frame.loc[mask,"target"].astype("int8").to_numpy(); p=predictions[f"{target}_{dataset}"]["calibrated"][mask]
  precision,recall,_=precision_recall_curve(y,p)
  ap=average_precision_score(y,p); ax.plot(recall,precision,label=f"{dataset} (AP={ap:.3f})")
 ax.axhline(float(test_frames[f"{target}_M0"].loc[test_frames[f"{target}_M0"].test_cohort.eq("test_seen"),"target"].mean()),color="gray",linestyle="--",label="Prevalence baseline")
 ax.set(title=f"{target}: test_seen precision–recall",xlabel="Recall",ylabel="Precision",xlim=(0,1),ylim=(0,1)); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_test_seen_precision_recall_curves.png",dpi=160,bbox_inches="tight"); plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 for dataset in ("M0","M1"):
  frame=test_frames[f"{target}_{dataset}"]; mask=frame.test_cohort.eq("test_seen").to_numpy()
  y=frame.loc[mask,"target"].astype("int8").to_numpy(); p=predictions[f"{target}_{dataset}"]["calibrated"][mask]
  fpr,tpr,_=roc_curve(y,p); auc=roc_auc_score(y,p); ax.plot(fpr,tpr,label=f"{dataset} (AUC={auc:.3f})")
 ax.plot([0,1],[0,1],"--",color="gray",label="Chance")
 ax.set(title=f"{target}: test_seen ROC",xlabel="False positive rate",ylabel="True positive rate",xlim=(0,1),ylim=(0,1)); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_test_seen_roc_curves.png",dpi=160,bbox_inches="tight"); plt.close(fig)

# Pointwise household-bootstrap intervals for final M0 metrics.
ci_metrics=("pr_auc","roc_auc","brier_score","precision","recall","f1")
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 part=bootstrap_ci.loc[(bootstrap_ci.target.eq(target))&(bootstrap_ci.quantity.eq("M0"))&bootstrap_ci.metric.isin(ci_metrics)].copy()
 part["metric"]=pd.Categorical(part.metric,categories=ci_metrics,ordered=True); part=part.sort_values("metric")
 yloc=np.arange(len(part)); mid=part.point_estimate.to_numpy(); lower=mid-part.ci_95_lower.to_numpy(); upper=part.ci_95_upper.to_numpy()-mid
 ax.errorbar(mid,yloc,xerr=np.vstack([lower,upper]),fmt="o",capsize=3)
 ax.set_yticks(yloc,part.metric.astype(str)); ax.invert_yaxis(); ax.set_xlim(0,1); ax.set(title=f"{target} M0 test_seen: 95% household-bootstrap CIs",xlabel="Metric")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_m0_metric_confidence_intervals.png",dpi=160,bbox_inches="tight"); plt.close(fig)

# Paired M1−M0 effects with household-bootstrap intervals; negative Brier/log-loss favors M1.
ci_delta_metrics=("pr_auc","roc_auc","brier_score","log_loss")
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,("P95","P90")):
 part=bootstrap_ci.loc[(bootstrap_ci.target.eq(target))&(bootstrap_ci.quantity.eq("M1-M0"))&bootstrap_ci.metric.isin(ci_delta_metrics)].copy()
 part["metric"]=pd.Categorical(part.metric,categories=ci_delta_metrics,ordered=True); part=part.sort_values("metric")
 yloc=np.arange(len(part)); mid=part.point_estimate.to_numpy(); lower=mid-part.ci_95_lower.to_numpy(); upper=part.ci_95_upper.to_numpy()-mid
 ax.errorbar(mid,yloc,xerr=np.vstack([lower,upper]),fmt="o",capsize=3); ax.axvline(0,color="gray",linestyle="--")
 ax.set_yticks(yloc,part.metric.astype(str)); ax.invert_yaxis(); ax.set(title=f"{target} paired test_seen M1−M0: 95% CI",xlabel="Difference (M1 minus M0)")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"08b_paired_weather_difference_confidence_intervals.png",dpi=160,bbox_inches="tight"); plt.close(fig)

p95_delta=bootstrap_ci.loc[(bootstrap_ci.target=="P95")&(bootstrap_ci.quantity=="M1-M0")&(bootstrap_ci.metric=="pr_auc")].iloc[0]
if p95_delta.ci_95_lower<=0<=p95_delta.ci_95_upper:
 weather_conclusion="No clear evidence that weather improves P95 discrimination: the paired 95% household-bootstrap interval for ΔPR-AUC includes zero."
else:
 weather_conclusion="The paired household-bootstrap interval for P95 ΔPR-AUC excludes zero; interpret its direction alongside the ROC-AUC, calibration, and threshold metrics."
print("P95 primary PR-AUC is average precision. Compare it with the observed prevalence baseline; it is not accuracy.")
print(weather_conclusion)
print("Natural unseen-household results are exploratory and should not support strong generalization claims.")


## 6. Validate and record the one-time final evaluation

In [ ]:
checks=[]
def add_check(name,passed,detail): checks.append({"check":name,"passed":bool(passed),"detail":str(detail)})
add_check("primary_full_m0_role_and_lock_valid",all(primary_bundles[t].get("model_role")=="primary_full_m0" and primary_lock["selected_models"][t]["model_role"]=="primary_full_m0" for t in ("P95","P90")),"Notebook 06 primary full-M0 bundles serve RQ2/RQ4")
add_check("paired_weather_role_is_isolated_to_rq3",all(bundles[k].get("model_role")=="controlled_weather_ablation" for k in bundles) and set(differences.model_role)=={"controlled_weather_ablation"},"Weather effect is paired M1 minus paired M0")
add_check("primary_bootstrap_ci_saved",(TABLE_DIR/"08b_primary_full_m0_household_bootstrap_ci.csv").exists() and len(primary_bootstrap_ci)>0,"Household bootstrap intervals for full-M0 models")
add_check("primary_all_outer_test_rows_scored",all(set(primary_test_frames[t].row_id.astype(str))==primary_test_ids for t in primary_test_frames),len(primary_test_ids))
add_check("development_prevalence_baseline_and_bss_reported",set(primary_metrics.target)=={"P95","P90"} and primary_metrics.loc[primary_metrics.cohort.eq("test_seen"),"brier_skill_score"].notna().all(),"Baseline uses development labels only")
add_check("primary_calibration_and_risk_outputs_saved",all((TABLE_DIR/name).exists() for name in ["08b_primary_full_m0_reliability_bins.csv","08b_primary_full_m0_calibration_diagnostics.csv","08b_primary_p95_risk_deciles.csv","08b_primary_p95_cumulative_gains.csv"]) and (FIGURE_DIR/"08b_primary_full_m0_reliability.png").exists() and (FIGURE_DIR/"08b_primary_p95_cumulative_gains.png").exists(),"Reliability, intercept/slope, deciles, gains")
add_check("evaluation_lock_manifest_passed",bool(lock.get("validation_passed")),"Notebook 08 lock was validated")
add_check("only_complete_weather_test_rows_loaded",all(set(f.row_id.astype(str))==test_ids and f.outer_split.eq("test").all() for f in test_frames.values()),len(test_ids))
add_check("requested_ids_are_outer_test_only",bool(test_master.outer_split.eq("test").all() and set(test_master.row_id.astype(str))==test_ids),"Only test IDs selected from the master split")
add_check("all_four_paired_locked_bundles_scored",set(predictions)=={"P95_M0","P95_M1","P90_M0","P90_M1"},sorted(predictions))
add_check("row_level_predictions_saved_for_local_explanations",row_prediction_path.exists() and len(row_predictions)==len(test_ids)*4+len(primary_test_ids)*2,len(row_predictions))
add_check("test_seen_primary_has_both_classes",all(test_frames[k].loc[test_frames[k].test_cohort.eq("test_seen"),"target"].nunique()==2 for k in test_frames),"test_seen ranking metrics are defined")
add_check("natural_unseen_cohort_is_small",int((reference.test_cohort=="test_unseen").sum())==277 and int(reference.loc[reference.test_cohort.eq("test_unseen"),"household_id"].nunique())==16,f"{int((reference.test_cohort=='test_unseen').sum())} rows")
add_check("paired_household_bootstrap_completed",bool(len(bootstrap_ci)>0 and bootstrap_ci.bootstrap_replicates.eq(N_BOOTSTRAP).all()),f"{N_BOOTSTRAP} replicates per interval")
add_check("final_metric_visuals_saved",all((FIGURE_DIR/name).exists() for name in ["08b_test_seen_precision_recall_curves.png","08b_test_seen_roc_curves.png","08b_m0_metric_confidence_intervals.png","08b_paired_weather_difference_confidence_intervals.png"]),"PR, ROC, M0 CIs, paired difference CIs")
add_check("no_fit_calibration_or_threshold_selection_in_final_stage",True,"Loaded frozen bundles; prediction only")
validation=pd.DataFrame(checks); display(validation); validation.to_csv(TABLE_DIR/"08b_final_validation.csv",index=False)
if not validation.passed.all(): raise AssertionError("Final test evaluation validation failed")
rq_summary=pd.DataFrame([{"research_question":"RQ2","result_source":"PRIMARY_P95_M0","model_role":"primary_full_m0","metrics_file":str(TABLE_DIR/"08b_primary_full_m0_metrics.csv")},{"research_question":"RQ4","result_source":"PRIMARY_P95_M0 and PRIMARY_P90_M0","model_role":"primary_full_m0","metrics_file":str(TABLE_DIR/"08b_primary_full_m0_metrics.csv")},{"research_question":"RQ3","result_source":"PAIRED_P95/P90_M0/M1","model_role":"controlled_weather_ablation","metrics_file":str(TABLE_DIR/"08b_paired_weather_differences.csv")},{"research_question":"RQ1","result_source":"Notebook 09 primary SHAP","model_role":"primary_full_m0","metrics_file":"notebooks/09_shap_analysis.ipynb"}])
rq_summary.to_csv(TABLE_DIR/"08b_result_to_research_question_map.csv",index=False)
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"stage":"08b_final_test_evaluation",
 "primary_models":{"role":"primary_full_m0","source":"Notebook 06","research_questions":["RQ2","RQ4"],"metrics_file":str(TABLE_DIR/"08b_primary_full_m0_metrics.csv"),"rows_scored":len(primary_test_ids)},
 "paired_weather_models":{"role":"controlled_weather_ablation","source":"Notebook 08","research_questions":["RQ3"],"difference_file":str(TABLE_DIR/"08b_paired_weather_differences.csv")},
 "result_to_research_question_map":str(TABLE_DIR/"08b_result_to_research_question_map.csv"),
 "first_outer_test_evaluation":True,"test_labels_loaded":True,"test_rows_scored":len(test_ids),
 "model_fitting_or_calibration_or_threshold_selection_performed":False,"paired_filter":"m1_weather_complete == True",
 "primary_cohort":"test_seen","exploratory_cohort":{"name":"natural test_unseen","households":16,"rows":277},
 "bootstrap":{"resampling_unit":"household","replicates":N_BOOTSTRAP,"seed":BOOTSTRAP_SEED,"paired_m0_m1":True},
 "selected_model_lock_manifest_sha256":sha256(LOCK_MANIFEST_PATH),
 "metrics_file":str(TABLE_DIR/"08b_final_test_metrics.csv"),"row_level_predictions_file":str(TABLE_DIR/"08b_final_test_row_predictions.csv"),"paired_differences_file":str(TABLE_DIR/"08b_paired_weather_differences.csv"),
 "bootstrap_intervals_file":str(TABLE_DIR/"08b_household_bootstrap_confidence_intervals.csv"),
 "primary_bootstrap_intervals_file":str(TABLE_DIR/"08b_primary_full_m0_household_bootstrap_ci.csv"),
 "validation_passed":bool(validation.passed.all()),"validation_checks":checks,"weather_interpretation":weather_conclusion,
 "metric_note":"PR-AUC is average precision and must not be interpreted as accuracy. test_unseen results are exploratory owing to 16 households and 277 rows."}
(TABLE_DIR/"08b_final_test_evaluation_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("FINAL OUTER-TEST EVALUATION COMPLETE — no modelling choices were changed after the lock.")


## Final interpretation

Use Notebook 06 full-M0 P95 as the primary RQ2 model and P90 as the RQ4 sensitivity model. Report PR-AUC as average precision, along with ROC-AUC, Brier score, log loss, threshold metrics, household-bootstrap intervals, and the development-prevalence Brier Skill Score. Reliability, calibration slope/intercept, and P95 lift/gains describe final-test behavior; they do not trigger recalibration or threshold changes.

Use paired Notebook 08 M1−M0 differences exclusively for RQ3 because both models use the same weather-supported population and identical test rows. Natural unseen-household results are supplementary and exploratory because this cohort has 16 households and 277 rows.